Este archivo detalla los analises utilizados en el capitulo 4

In [14]:
#### Importando modulos 
import pandas as pd
import numpy as np
import scipy
from scipy import stats
import matplotlib.pyplot as plt

In [15]:
##### Funcion para integrar posiciones
def posiciones(df_r: pd.DataFrame, empresas: list[str], tipo:str):
    df_posiciones = pd.DataFrame()
    #### Extracciones individuales
    if tipo=='sliding':
        for i in range(22):
            tmp = scipy.io.loadmat(f'SlidingFinalPreds/pred_sliding_{i}.mat')
            df_tmp = pd.DataFrame(tmp['Ypred_final'])
            df_posiciones = pd.concat([df_posiciones, df_tmp], axis=0, ignore_index=True)
        del tmp
        df_posiciones.columns = [f'Posicion.{nom}' for nom in empresas]
    elif tipo=='expanding':
        for i in range(22):
            tmp = scipy.io.loadmat(f'ExpandingFinalPreds/pred_expanding_{i}.mat')
            df_tmp = pd.DataFrame(tmp['Ypred_final'])
            df_posiciones = pd.concat([df_posiciones, df_tmp], axis=0, ignore_index=True)
        del tmp
        df_posiciones.columns = [f'Posicion.{nom}' for nom in empresas]
    elif tipo=='todo':
        for nom in empresas:
            df_posiciones[f'Posicion.{nom}'] = [1] * df_r.shape[0]
    elif tipo=='rand':
        import random
        random.seed(42)
        for nom in empresas:
            df_posiciones[f'Posicion.{nom}'] = [random.randint(0,1) for _ in range(df_r.shape[0])]


    #### Asegurando que df_posiciones tenga mismo numero de datos
    df_posiciones = df_posiciones.iloc[:df_r.shape[0]]

    #### Camiando indices
    df_posiciones.index = df_r.index
    
    #### Sumando cantidad de posiciones
    suma = df_posiciones.sum(axis=1)

    #### Asignando posiciones equitativamente
    df_posiciones_finales = df_posiciones.div(suma, axis=0)

    #### Integrando posiciones con rendimientos
    df_integrado = pd.concat([df_r,df_posiciones_finales], axis=1, ignore_index=False)

    return df_integrado


In [16]:
##### Calculando formula para la rentabilidad del portafolio

def calc_rentabilidad(df:pd.Dataframe, empresas: list[str], tgt: int = 0.1, costo : int = 0.00001) -> pd.Dataframe:
    """
    Formula para calcular la rentabilidad de un portafolio 

    Params:
    df: el dataframe que contiene los rendimientos, EWMSD, y posiciones del portafolio
    tgt: la voltatilidad deseada para escalar la volatilidad proporcionalmente
    costo: costo de cambiar las posiciones de las acciones en un portafolio

    Return:
    df_rentabilidad: el df que contiene la rentabilidad del portafolio por las fechas marcadas
    """
    #### Calculando rentabilidad equitativa por cada accion/empresa
    df_rentabilidad = pd.DataFrame()
    for name in empresas:
        df_rentabilidad[f'Rentabilidad.{name}'] = df[name] * (tgt/df[f'EWMSD.{name}'].shift(1))  * df[f'Posicion.{name}'].shift(1) - \
        costo*abs(((tgt/df[f'EWMSD.{name}'].shift(1))*df[f'Posicion.{name}'].shift(1)) - ((tgt/df[f'EWMSD.{name}'].shift(2))*df[f'Posicion.{name}'].shift(2)))
        # df_rentabilidad[f'Rentabilidad.{name}'] = df[name] * df[f'Posicion.{name}']


    #### Elimando primera dos fila por el "shift(2)"
    df_rentabilidad_diaria = df_rentabilidad.iloc[2:]

    return df_rentabilidad_diaria



In [17]:
#### Formula para limpiar datos
def cargar(df_original: pd.DataFrame, tipo: str):
    ### Copiando datos para no malograr
    df = df_original.copy()

    ### Nombre de las 32 empresas
    empresas = df.columns.to_list()

    ### Calculando EWMSD
    for name in empresas:
        df[f'EWMSD.{name}'] = df[name].ewm(span=30, adjust=False).std()

    ### Eliminando datos
    df = df.iloc[:-62]
    df = df.iloc[345:]

    #### Calculando posiciones
    df_posiciones= posiciones(df, empresas, tipo)    

    #### Calculado rentabilidades diarias de todas acciones
    df_rentabilidad_diaria = calc_rentabilidad(df_posiciones, empresas)
    
    return df_rentabilidad_diaria

In [18]:
#### Funcion para extraer estadisticas
def estadisticas(df:pd.DataFrame):
    promedio = df.mean()
    mediana = df.median()
    des_est = df.std(ddof=0)
    minimo = df.min()
    maximo = df.max()

    return [promedio, mediana, des_est, minimo, maximo]

In [ ]:
#### Funcion para extraer estadisticas de rendimiento esperado, vol, y ratio sharpe
def extraccion(df: pd.DataFrame):
    df_rendis_esperado = df.mean(axis=0)
    df_vol = df.std(axis=0, ddof=0)
    df_ratio_sharpe = df_rendis_esperado/(df_vol + 1e-8)

    return estadisticas(df_rendis_esperado), estadisticas(df_vol), estadisticas(df_ratio_sharpe)

In [20]:
df_rendis = pd.read_csv('retornos.csv', index_col='Date')

df_rand_rentabilidad = cargar(df_rendis, 'rand')
df_lstm_rentabilidad = cargar(df_rendis, 'sliding')

rand_rendis_estads, rand_vol_estads, rand_ratio_estads = extraccion(df_rand_rentabilidad)

Rentabilidad.ABEV         0.004749
Rentabilidad.AC           0.004697
Rentabilidad.AMXB         0.006733
Rentabilidad.AXIA3        0.004940
Rentabilidad.BAP          0.004923
Rentabilidad.BBAS3        0.004841
Rentabilidad.BBD          0.005050
Rentabilidad.BIMBOA       0.004789
Rentabilidad.BSAC         0.004918
Rentabilidad.CEMEXCPO     0.004715
Rentabilidad.CENCOSUD     0.004925
Rentabilidad.CIB          0.004752
Rentabilidad.FALABELLA    0.004852
Rentabilidad.FEMSAUBD     0.004712
Rentabilidad.GCARSOA1     0.006328
Rentabilidad.GFNORTEO     0.004798
Rentabilidad.GGB          0.004750
Rentabilidad.GMEXICOB     0.004748
Rentabilidad.ISA          0.005022
Rentabilidad.ITSA4        0.004790
Rentabilidad.ITUB         0.004878
Rentabilidad.PAC          0.005008
Rentabilidad.PBR          0.004831
Rentabilidad.PBR-A        0.004920
Rentabilidad.RENT3        0.004612
Rentabilidad.SBSP3        0.005014
Rentabilidad.SCCO         0.004705
Rentabilidad.SQM          0.004783
Rentabilidad.VALE   

Rentabilidad.ABEV         0.005794
Rentabilidad.AC           0.013868
Rentabilidad.AMXB         0.013165
Rentabilidad.AXIA3        0.007730
Rentabilidad.BAP          0.024795
Rentabilidad.BBAS3        0.024289
Rentabilidad.BBD         -0.006411
Rentabilidad.BIMBOA       0.021455
Rentabilidad.BSAC         0.022385
Rentabilidad.CEMEXCPO     0.009143
Rentabilidad.CENCOSUD    -0.004632
Rentabilidad.CIB          0.021190
Rentabilidad.FALABELLA    0.022838
Rentabilidad.FEMSAUBD     0.009001
Rentabilidad.GCARSOA1     0.010715
Rentabilidad.GFNORTEO     0.019929
Rentabilidad.GGB         -0.001087
Rentabilidad.GMEXICOB     0.023068
Rentabilidad.ISA          0.030953
Rentabilidad.ITSA4        0.036112
Rentabilidad.ITUB         0.006986
Rentabilidad.PAC          0.011006
Rentabilidad.PBR          0.017345
Rentabilidad.PBR-A        0.017592
Rentabilidad.RENT3        0.015673
Rentabilidad.SBSP3        0.026077
Rentabilidad.SCCO         0.030776
Rentabilidad.SQM          0.018566
Rentabilidad.VALE   